# Agentic Document Extractor - Interactive Demo

This notebook demonstrates the end-to-end document extraction pipeline for Indian government and administrative documents.

## Features
- Multi-language OCR (Hindi + English)
- Layout detection with reading order
- VLM-based table, form, and stamp analysis
- Schema-based structured extraction

## Setup

First, ensure you have the required environment variables set:

In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

# Check for OpenAI API key
if not os.getenv("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not set. VLM features will be disabled.")
    print(f"Using {OPENAI_API_KEY}")
else:
    print("OpenAI API key found!")

## Import the library

In [6]:
from agentic_document_extractor import (
    DocumentProcessor,
    ProcessingResult,
    VOTER_LIST_SCHEMA,
    AGENT_DETAILS_SCHEMA,
    draw_layout,
    create_comparison_view,
)
from agentic_document_extractor.core.dataclasses import RegionType
from agentic_document_extractor.utils.helpers import load_document, save_results_csv
from agentic_document_extractor.utils.visualization import generate_html_report

import matplotlib.pyplot as plt
from PIL import Image

# Configure matplotlib
plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['figure.dpi'] = 150

## Initialize the Document Processor

The processor handles OCR, layout detection, and VLM-based analysis.

In [ ]:
# Initialize processor with Hindi and English support
processor = DocumentProcessor(
    ocr_languages=("hi", "en"),  # Hindi + English
    use_gpu=False,                # Set to True if GPU available
    verbose=True,                 # Show processing logs
    dpi=150,                      # Document conversion DPI
    max_pages=10,                 # Maximum pages to process
)

print("Document Processor initialized!")
print(f"OCR Languages: {processor.ocr_languages}")
print(f"VLM available: {processor.vlm_client is not None}")

## Load a Sample Document

You can use:
- PDF files (`.pdf`)
- Image files (`.png`, `.jpg`, `.tiff`)
- Word documents (`.docx`)

In [ ]:
# Path to your document
DOCUMENT_PATH = "../examples/sample_documents/voter_list.pdf"  # Replace with your document

# Check if document exists, otherwise create a sample image
import os
if not os.path.exists(DOCUMENT_PATH):
    print(f"Document not found: {DOCUMENT_PATH}")
    print("Creating a sample image for demonstration...")
    
    # Create a sample image that looks like a document
    sample_img = Image.new('RGB', (800, 600), color='white')
    from PIL import ImageDraw, ImageFont
    draw = ImageDraw.Draw(sample_img)
    
    # Add some text lines
    draw.text((50, 50), "Sample Voter List", fill='black')
    draw.text((50, 100), "Constituency: Lucknow East", fill='black')
    draw.text((50, 150), "Serial | Name | Age | Gender | Address", fill='black')
    draw.text((50, 200), "1 | Ramesh Kumar | 45 | M | Village Ramnagar", fill='black')
    draw.text((50, 230), "2 | Sunita Devi | 42 | F | Village Ramnagar", fill='black')
    
    # Save sample
    os.makedirs("../examples/sample_documents", exist_ok=True)
    sample_img.save("../examples/sample_documents/sample_voter_list.png")
    DOCUMENT_PATH = "../examples/sample_documents/sample_voter_list.png"

print(f"Using document: {DOCUMENT_PATH}")

# Load and display the document
images = load_document(DOCUMENT_PATH)
print(f"Loaded {len(images)} page(s)")

# Show first page
plt.imshow(images[0])
plt.axis('off')
plt.tight_layout()
plt.show()

## Process the Document

This runs the full pipeline:
1. **OCR**: Extract text with Hindi/English support
2. **Layout Detection**: Identify tables, forms, stamps, etc.
3. **Region Analysis**: VLM-based analysis of complex regions

In [ ]:
%%time

# Process the document
result = processor.process(
    DOCUMENT_PATH,
    analyze_regions=True,  # Enable VLM analysis
)

print(f"\n{'='*50}")
print(f"Processing Complete!")
print(f"{'='*50}")
print(f"Pages processed: {result.page_count}")
print(f"Processing time: {result.processing_time:.2f} seconds")
print(f"Layout regions detected: {len(result.layouts[0].regions)}")
print(f"Errors: {len(result.errors)}")

## Visualize Layout Detection

See how the document was analyzed and regions were identified.

In [ ]:
# Create visualization
layout = result.layouts[0]
image = result.images[0] if result.images else images[0]

# Side-by-side comparison
comparison = create_comparison_view(image, layout)

plt.figure(figsize=(20, 8))
plt.imshow(comparison)
plt.axis('off')
plt.tight_layout()
plt.show()

# Print region summary
print("\nDetected Regions:")
print("-" * 40)
for i, region in enumerate(layout.regions):
    text_preview = region.combined_text[:50] if region.combined_text else "(no text)"
    print(f"{i+1}. [{region.region_type.value}] Order: {region.reading_order}")
    print(f"   Text: {text_preview}...")

## Extract Structured Data

Use predefined schemas to extract structured data.

In [ ]:
# Extract voter list data
voter_data = processor.extract_voter_list(result)

if "error" in voter_data:
    print(f"Extraction note: {voter_data['error']}")
else:
    print(f"\nExtracted {len(voter_data.get('voters', []))} voters")
    
    # Display first few voters
    for i, voter in enumerate(voter_data.get('voters', [])[:3]):
        print(f"\nVoter {i+1}:")
        for key, value in voter.items():
            if value is not None:
                print(f"  {key}: {value}")

## Custom Schema Extraction

You can also use custom JSON schemas for specific document types.

In [ ]:
# Use agent details schema
agent_data = processor.extract_agent_details(result)

if "error" not in agent_data:
    print("Agent Details Extraction:")
    print("-" * 40)
    for key, value in agent_data.items():
        if value is not None and key not in ('confidence', 'notes'):
            print(f"{key}: {value}")

## Analyze Specific Regions

Use the orchestrator to analyze specific region types (tables, forms, stamps).

In [ ]:
# Check if orchestrator is available
if processor.orchestrator:
    # Analyze only table regions
    analyses = processor.orchestrator.analyze_regions(
        layout=layout,
        image=image,
        region_types=[RegionType.TABLE, RegionType.FORM],
    )
    
    print("Region Analyses:")
    print("-" * 40)
    for region_id, analysis in analyses.items():
        print(f"\nRegion {region_id}:")
        print(f"  Type: {analysis.get('region_type')}")
        if 'analysis' in analysis:
            result_data = analysis['analysis']
            if isinstance(result_data, dict):
                for k, v in list(result_data.items())[:3]:
                    print(f"  {k}: {v}")
else:
    print("Orchestrator not available (VLM client not configured)")

## Save Results

Export results in various formats.

In [ ]:
import json
from pathlib import Path

# Create output directory
output_dir = Path("../examples/outputs")
output_dir.mkdir(exist_ok=True)

# Save full results as JSON
result.save_json(output_dir / "full_results.json")
print(f"Saved: {output_dir / 'full_results.json'}")

# Save voter data as CSV
if 'voters' in voter_data:
    save_results_csv(voter_data['voters'], output_dir / "voters.csv")
    print(f"Saved: {output_dir / 'voters.csv'}")

# Generate HTML report
generate_html_report(
    layout=layout,
    extracted_data=voter_data,
    output_path=output_dir / "report.html",
)
print(f"Saved: {output_dir / 'report.html'}")

## Batch Processing

Process multiple documents in a directory.

In [ ]:
from pathlib import Path

def batch_process_documents(input_dir, output_dir, schema="voter_list"):
    """Process all documents in a directory."""
    input_path = Path(input_dir)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # Find all supported documents
    extensions = ["*.pdf", "*.png", "*.jpg", "*.jpeg", "*.tiff", "*.docx"]
    documents = []
    for ext in extensions:
        documents.extend(input_path.glob(ext))
    
    print(f"Found {len(documents)} documents")
    
    results = []
    for doc_path in documents:
        print(f"\nProcessing: {doc_path.name}")
        try:
            result = processor.process(doc_path)
            extraction = processor.extract_schema(result, schema)
            
            # Save individual results
            doc_output = output_path / doc_path.stem
            doc_output.mkdir(exist_ok=True)
            result.save_json(doc_output / "results.json")
            
            results.append({
                "document": doc_path.name,
                "pages": result.page_count,
                "extraction": extraction,
            })
        except Exception as e:
            print(f"  Error: {e}")
    
    # Save batch summary
    with open(output_path / "batch_summary.json", "w") as f:
        json.dump(results, f, indent=2)
    
    print(f"\nBatch complete! Results in: {output_path}")
    return results

# Example usage (uncomment to run):
# batch_results = batch_process_documents(
#     "../examples/sample_documents",
#     "../examples/batch_outputs"
# )

## Performance Analysis

In [ ]:
# Show processing statistics
print("Processing Statistics:")
print("=" * 40)
print(f"Total pages: {result.page_count}")
print(f"Total time: {result.processing_time:.2f}s")
print(f"Time per page: {result.processing_time / max(result.page_count, 1):.2f}s")
print(f"\nRegion breakdown:")

from collections import Counter
region_types = Counter(r.region_type.value for r in layout.regions)
for rt, count in region_types.items():
    print(f"  {rt}: {count}")

## Cleanup

In [ ]:
# Clear region cache to free memory
processor.region_processor.clear_cache()
print("Cache cleared!")

# Show final cache stats
stats = processor.region_processor.get_cache_stats()
print(f"Cache stats: {stats}")